# **` Step 1 - Imorting Libarires `**

## Multi-class Classification ( 3 Classes ) 
## Modelling & Evalustion 

In [ ]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras import layers
from tensorflow.keras.layers import (
    Rescaling,Conv2D, MaxPooling2D,
    Flatten, Dense, Dropout
)
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint


import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix



# **` Step 2 - Loading Image dataset from Directory `**

In [ ]:
data_dir = "/kaggle/input/covid19-pneumonia-normal-chest-xray-pa-dataset"
data_dir

In [ ]:
train_ds = image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(224, 224),
    batch_size=32
)

val_ds = image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(224, 224),
    batch_size=32
)


In [ ]:
5522+1380

In [ ]:
6902 * 0.2

In [ ]:
(224*224 ) * 3 # ( 0 : 255 )

In [ ]:
train_ds

In [ ]:
val_ds # test

In [ ]:
class_names = train_ds.class_names
class_names


# **` Step 3 - Image Preprocessing ( Normalization ) `**

In [ ]:
normalization_layer = Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds   = val_ds.map(lambda x, y: (normalization_layer(x), y))


In [ ]:
train_ds

In [ ]:
images, labels = next(iter(train_ds))# take 1 batch
plt.figure(figsize=(10, 10))

for i in range(9):
    plt.subplot(3, 3, i+1)              # 3x3 grid
    plt.imshow(images[i])               # show image
    plt.title(class_names[int(labels[i])])  # show label from folder name
    plt.axis("off")

plt.show()


# **` Step 4 - Image Augmentation `**

In [ ]:
augment = Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.2)
])
augment

In [ ]:
train_ds

In [ ]:
val_ds

# **` Step 5 - Sequential Model Bulding (Augmenation + CNN + ANN ) `**

In [ ]:
model = Sequential([
    
    layers.Input(shape=(224, 224, 3)),
    
    augment, #Image ( Augmentation )
    
    Conv2D(32, 3 , activation="relu" ), # 32 kernal ( filter ( 3*3 ))
    MaxPooling2D(),

    Conv2D(64, 3, activation="relu"),
    MaxPooling2D(),

    Conv2D(128, 3, activation="relu"),
    MaxPooling2D(),


    Flatten(),

    Dense(128, activation="relu"),
    Dropout(0.5),

    Dense(64, activation="relu"),
    Dropout(0.5),

    Dense(32, activation="relu"),
    Dropout(0.5),


    Dense(3, activation="softmax")
])


# **` Step 6 - Model Compilation `**

In [ ]:
32 * 3 * 3 *3 + 32

In [ ]:
( 32 * 3 * 3 + 1 ) * 64

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

In [ ]:
26*26*128

In [ ]:
86528 * 128

# **` Step 7 - Model CallBacks `**

In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6),
    ModelCheckpoint("/kaggle/working/best_model.keras", save_best_only=True, monitor="val_loss")
]



In [ ]:
5522 / 32

# **` Step 8 - Model Training `**

In [ ]:
0.001 * 0.5

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks
)


# **` Step 9 - Plotting Model Performance `**

In [ ]:
history

In [ ]:
plt.figure(figsize=(12, 5))


plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Val")
plt.title("Accuracy")
plt.legend()


plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Val")
plt.title("Loss")
plt.legend()

plt.show()


# **` Step 10 - Making a prediction ! `**

In [ ]:
images, labels = next(iter(val_ds))   # 1 batch from test
idx = np.random.randint(0, len(images))

img = images[idx]
true_label = int(labels[idx])

plt.imshow(img)
plt.title("True: " + class_names[true_label])
plt.axis("off")
plt.show()

pred = model.predict(np.expand_dims(img, axis=0), verbose=0)
pred_label = np.argmax(pred)

print("Predicted:", class_names[pred_label])
print("Probabilities:", {class_names[i]: float(pred[0][i]) for i in range(3)})


# **` Step 11 - Model Evaluation ! `**

In [ ]:
test_loss, test_acc = model.evaluate(val_ds)
print("Test Accuracy:", test_acc)


In [ ]:
1380 / 32

In [ ]:
import numpy as np

y_true = []
y_pred = []

for images, labels in val_ds:
    predictions = model.predict(images, verbose=1)
    predicted_labels = np.argmax(predictions, axis=1)

    y_true.extend(labels.numpy())
    y_pred.extend(predicted_labels)


In [ ]:
print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))


In [ ]:
import json 
with open("/kaggle/working/class_names.json" , "w") as f : 
    json.dump(class_names, f ) 

print("SAVED : /kaggle/working/class_names.json")
    

# **` See You in Next Chapter  !! `**